# From fermions to qubits: the Jordan-Wigner transformation

Executable companion to the last section of chapter 3 and to the
qubit-encoding section of chapter 4.

A quantum computer acts on qubits, not on fermionic Fock states.  The
occupation-number basis is already binary, so the *states* map across
trivially,

$$
|n_0 n_1\cdots n_{M-1}\rangle_{\rm Fock}
\;\longleftrightarrow\;
|n_0 n_1\cdots n_{M-1}\rangle_{\rm qubit},
$$

and all the work is in the *operators*: the fermionic minus signs have to be
reproduced by something.  The Jordan-Wigner transformation supplies them with
a string of Pauli $Z$'s,

$$
a_p^\dagger = \left(\prod_{j<p} Z_j\right)\frac{X_p - iY_p}{2},
\qquad
a_p = \left(\prod_{j<p} Z_j\right)\frac{X_p + iY_p}{2}.
$$

Everything below is done with explicit matrices, so nothing has to be taken
on trust.

In [ ]:
from itertools import product

import numpy as np

I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
PAULI = {"I": I2, "X": X, "Y": Y, "Z": Z}


def kron(matrices):
    out = np.array([[1.0 + 0j]])
    for matrix in matrices:
        out = np.kron(out, matrix)
    return out


def pauli_string(label):
    return kron([PAULI[c] for c in label])


def single(op, site, n_qubits):
    return kron([op if k == site else I2 for k in range(n_qubits)])


def decompose(matrix, tol=1e-10):
    """H = sum_P c_P P with c_P = Tr(P H)/2^M -- exact, by orthogonality."""
    dim = matrix.shape[0]
    n_qubits = int(round(np.log2(dim)))
    out = {}
    for word in product("IXYZ", repeat=n_qubits):
        label = "".join(word)
        c = np.trace(pauli_string(label) @ matrix) / dim
        if abs(c) > tol:
            out[label] = float(c.real) if abs(c.imag) < tol else complex(c)
    return out


def show(d, indent="   "):
    for label, c in sorted(d.items(), key=lambda kv: (-abs(kv[1]), kv[0])):
        print(f"{indent}{c:+.6f}  {label}")


def weight(label):
    return sum(1 for c in label if c != "I")

In [ ]:
class JordanWigner:
    """Fermion operators on n modes, as 2^n x 2^n qubit matrices."""

    def __init__(self, n_modes):
        self.n = n_modes
        self.dim = 1 << n_modes

    def _string(self, p):
        return kron([Z if k < p else I2 for k in range(self.n)])

    def create(self, p):
        return self._string(p) @ single((X - 1j*Y)/2.0, p, self.n)

    def annihilate(self, p):
        return self.create(p).conj().T

    def number(self, p):
        return self.create(p) @ self.annihilate(p)

    def identity(self):
        return np.eye(self.dim, dtype=complex)

    def term(self, coefficient, creators, annihilators):
        out = coefficient * self.identity()
        for p in creators:
            out = out @ self.create(p)
        for p in annihilators:
            out = out @ self.annihilate(p)
        return out

    def anticommutator_errors(self):
        worst, eye = 0.0, self.identity()
        for p in range(self.n):
            for q in range(self.n):
                a_p, a_q, c_p, c_q = (self.annihilate(p), self.annihilate(q),
                                      self.create(p), self.create(q))
                target = eye if p == q else 0.0*eye
                worst = max(worst, np.abs(a_p@c_q + c_q@a_p - target).max())
                worst = max(worst, np.abs(a_p@a_q + a_q@a_p).max())
                worst = max(worst, np.abs(c_p@c_q + c_q@c_p).max())
        return float(worst)

## 1. Is it a legal transformation?

The whole point of the parity string is to make the qubit operators
*anticommute*.  Without it they would merely commute on different sites,
which is the wrong statistics.

In [ ]:
for n in (2, 3, 4, 5):
    print(f"{n} modes: max violation of the anticommutators = "
          f"{JordanWigner(n).anticommutator_errors():.1e}")

n = 3
naive = [single((X + 1j*Y)/2.0, p, n) for p in range(n)]
print("\nwithout the parity string, |{sigma_0, sigma_1}| =",
      np.abs(naive[0]@naive[1] + naive[1]@naive[0]).max(), " (should be 0)")

## 2. Number operators, hopping, and pairs

Three identities do all the work:

$$
\hat n_p = \frac{I-Z_p}{2},
$$
$$
a_p^\dagger a_q + a_q^\dagger a_p \;\to\;
\tfrac12\left(X_pZ\cdots ZX_q + Y_pZ\cdots ZY_q\right),
$$
$$
a_p^\dagger a_q^\dagger + a_q a_p \;\to\;
\tfrac12\left(X_pZ\cdots ZX_q - Y_pZ\cdots ZY_q\right).
$$

The last two differ only in one sign — that single sign is the difference
between transporting a particle and creating a pair.

In [ ]:
jw = JordanWigner(4)

print("n_p:")
for p in range(4):
    print("  ", decompose(jw.number(p)))

print("\nhopping  a^+_p a_q + a^+_q a_p:")
for p, q in ((0, 1), (0, 2), (0, 3)):
    print(f"  (p, q) = ({p}, {q}):")
    show(decompose(jw.create(p)@jw.annihilate(q)
                   + jw.create(q)@jw.annihilate(p)), "     ")

print("\npair  a^+_p a^+_q + a_q a_p:")
for p, q in ((0, 1), (0, 3)):
    print(f"  (p, q) = ({p}, {q}):")
    show(decompose(jw.create(p)@jw.create(q)
                   + jw.annihilate(q)@jw.annihilate(p)), "     ")

## 3. A general two-body term

With four distinct indices the Hermitian two-body term becomes eight Pauli
strings of weight four, each with coefficient $1/8$.  Read the four orbitals
as $(p_+, p_-, q_+, q_-)$ and this operator is the pair-transfer term
$P_p^\dagger P_q + {\rm h.c.}$ of the pairing model.

In [ ]:
matrix = jw.term(1.0, [0, 1], [3, 2])          # a^+_0 a^+_1 a_3 a_2
hermitian = matrix + matrix.conj().T
show(decompose(hermitian))
counts = {}
for label in decompose(hermitian):
    counts[weight(label)] = counts.get(weight(label), 0) + 1
print("strings by weight:", dict(sorted(counts.items())))

For a generic one- plus two-body Hamiltonian on $M$ modes the number of
distinct Pauli strings grows as $O(M^4)$, and the longest of them span the
whole register because of the parity chains.

In [ ]:
def one_body(jw, h):
    out = np.zeros((jw.dim, jw.dim), dtype=complex)
    for p in range(jw.n):
        for q in range(jw.n):
            out += h[p, q] * jw.create(p) @ jw.annihilate(q)
    return out


def two_body(jw, v):
    out = np.zeros((jw.dim, jw.dim), dtype=complex)
    for p, q, r, s in product(range(jw.n), repeat=4):
        if v[p, q, r, s] != 0.0:
            out += 0.25*v[p, q, r, s] * (jw.create(p) @ jw.create(q)
                                         @ jw.annihilate(s) @ jw.annihilate(r))
    return out


rng = np.random.default_rng(11)
print(f"{'M':>4s} {'4^M':>8s} {'strings':>9s} {'max weight':>12s}")
for M in (2, 3, 4, 5, 6):
    jw_m = JordanWigner(M)
    h = rng.normal(size=(M, M)); h = h + h.T
    v = rng.normal(size=(M,)*4)
    v = v - v.transpose(1, 0, 2, 3)
    v = v - v.transpose(0, 1, 3, 2)
    v = v + v.transpose(2, 3, 0, 1)
    d = decompose(one_body(jw_m, h) + two_body(jw_m, v))
    print(f"{M:4d} {4**M:8d} {len(d):9d} "
          f"{max(weight(k) for k in d):12d}")

## 4. The pairing model, two ways

Order the spin-orbitals so the two partners of a level are adjacent,
$(1_+, 1_-, 2_+, 2_-, \dots)$.  With $L$ levels that is $2L$ qubits.

In [ ]:
def pairing_full_space(levels, g, xi=1.0):
    jw = JordanWigner(2*levels)
    H = np.zeros((jw.dim, jw.dim), dtype=complex)
    for p in range(1, levels+1):
        for spin in (0, 1):
            H += xi*(p-1) * jw.number(2*(p-1) + spin)
    for p in range(1, levels+1):
        for q in range(1, levels+1):
            H += jw.term(-0.5*g, [2*(p-1), 2*(p-1)+1],
                         [2*(q-1)+1, 2*(q-1)])
    return H


def pairing_ph_full_space(levels, g, f, xi=1.0):
    jw = JordanWigner(2*levels)
    H = pairing_full_space(levels, g, xi)
    for p in range(1, levels+1):
        for q in range(1, levels+1):
            for r in range(1, levels+1):
                block = jw.term(-0.5*f, [2*(p-1), 2*(p-1)+1],
                                [2*(q-1)+1, 2*(r-1)])
                H += block + block.conj().T
    return H


H_full = pairing_full_space(2, g=1.0)
d = decompose(H_full)
print(f"L = 2 -> 4 qubits, {len(d)} Pauli strings:")
show(d)

In the seniority-zero space a level is either empty or holds a complete pair,
so its state is one bit and not two.  With the local quasispin operators
$S_p^+ = P_p^\dagger$, $S_p^z = (N_p-1)/2$ identified as
$S_p^+ = (X_p - iY_p)/2$ and $N_p = I - Z_p$, the Hamiltonian collapses to

$$
\hat H = \sum_{p=1}^{L}\left[\xi(p-1)-\frac{g}{4}\right](I - Z_p)
  - \frac{g}{4}\sum_{p<q}\left(X_pX_q + Y_pY_q\right)
$$

on $L$ qubits: an $XY$ model with infinite-range couplings in an
inhomogeneous field.

In [ ]:
def pairing_pair_qubits(levels, g, xi=1.0):
    dim = 1 << levels
    H = np.zeros((dim, dim), dtype=complex)
    plus = [single((X - 1j*Y)/2.0, p, levels) for p in range(levels)]
    minus = [single((X + 1j*Y)/2.0, p, levels) for p in range(levels)]
    number = [np.eye(dim, dtype=complex) - single(Z, p, levels)
              for p in range(levels)]
    for p in range(levels):
        H += xi*p*number[p]
        for q in range(levels):
            H += -0.5*g * plus[p] @ minus[q]
    return H


def closed_form(levels, g, xi=1.0):
    out = {}

    def add(label, value):
        out[label] = out.get(label, 0.0) + value

    for p in range(levels):
        c = xi*p - 0.25*g
        add("I"*levels, c)
        add("".join("Z" if k == p else "I" for k in range(levels)), -c)
    for p in range(levels):
        for q in range(p+1, levels):
            for letter in "XY":
                add("".join(letter if k in (p, q) else "I"
                            for k in range(levels)), -0.25*g)
    return {k: v for k, v in out.items() if abs(v) > 1e-12}


H_pair = pairing_pair_qubits(2, g=1.0)
d = decompose(H_pair)
print(f"L = 2 -> 2 qubits, {len(d)} Pauli strings:")
show(d)
for L in (2, 3, 4):
    predicted, computed = closed_form(L, 1.0), decompose(pairing_pair_qubits(L, 1.0))
    ok = (set(predicted) == set(computed)
          and all(abs(predicted[k]-computed[k]) < 1e-10 for k in predicted))
    print(f"   L = {L}: closed form reproduces the decomposition: {ok}")

The compact encoding is a **restriction, not an approximation**: its spectrum
is exactly the seniority-zero part of the full one, at fixed particle number.

In [ ]:
def seniority_zero_indices(levels):
    keep = []
    for state in range(1 << (2*levels)):
        if all(((state >> (2*p)) & 1) == ((state >> (2*p+1)) & 1)
               for p in range(levels)):
            keep.append(state)
    return keep


def particle_number(n_modes):
    return np.array([bin(s).count("1") for s in range(1 << n_modes)])


for levels in (2, 3):
    H_full = pairing_full_space(levels, g=1.0)
    H_pair = pairing_pair_qubits(levels, g=1.0)
    keep, occ = seniority_zero_indices(levels), particle_number(2*levels)
    for pairs in range(1, levels+1):
        rows = [k for k in keep if occ[k] == 2*pairs]
        a = np.sort(np.linalg.eigvalsh(H_full[np.ix_(rows, rows)]).real)
        pr = [k for k in range(1 << levels) if bin(k).count("1") == pairs]
        b = np.sort(np.linalg.eigvalsh(H_pair[np.ix_(pr, pr)]).real)
        print(f"L = {levels}, {pairs} pair(s): {len(rows):3d} states, "
              f"agree: {np.allclose(a, b)},  E_0 = {a[0]:.8f}")

## 5. What the particle-hole term costs

The particle-hole term breaks pairs, so the seniority-zero space is no longer
invariant and the one-qubit-per-level encoding is not available at all.  The
leakage out of that space is exactly $f/2$, the $1p$–$1h$ matrix element of
chapter 4.

In [ ]:
levels = 2
for f in (0.0, 0.1):
    H = pairing_ph_full_space(levels, g=1.0, f=f)
    d = decompose(H)
    counts = {}
    for label in d:
        counts[weight(label)] = counts.get(weight(label), 0) + 1
    keep, occ = seniority_zero_indices(levels), particle_number(2*levels)
    rows = [k for k in keep if occ[k] == 2]
    others = [k for k in range(1 << (2*levels))
              if occ[k] == 2 and k not in rows]
    print(f"f = {f}: {len(d):3d} Pauli strings, weights "
          f"{dict(sorted(counts.items()))}")
    print(f"        largest element out of the seniority-zero space: "
          f"{np.abs(H[np.ix_(others, rows)]).max():.4f}")

print()
print(f"{'model':>22s} {'qubits':>8s} {'strings':>9s} {'max weight':>12s}")
for levels in (2, 3):
    for name, matrix in (("pairing, pair qubits",
                          pairing_pair_qubits(levels, g=1.0)),
                         ("pairing, full space",
                          pairing_full_space(levels, g=1.0)),
                         ("pairing + p-h, full",
                          pairing_ph_full_space(levels, g=1.0, f=0.1))):
        d = decompose(matrix)
        n_qubits = int(round(np.log2(matrix.shape[0])))
        print(f"{name:>22s} {n_qubits:8d} {len(d):9d} "
              f"{max(weight(k) for k in d):12d}")

## 6. Nothing is lost in translation

The qubit Hamiltonian is the same operator written differently, so it has to
give the energies of chapter 4 exactly.  $L=4$, $N=4$, $g=1$, in the
$S_z = 0$ sector:

In [ ]:
def two_sz(levels):
    out = []
    for state in range(1 << (2*levels)):
        up = sum((state >> (2*p)) & 1 for p in range(levels))
        dn = sum((state >> (2*p+1)) & 1 for p in range(levels))
        out.append(up - dn)
    return np.array(out)


occ, spin = particle_number(8), two_sz(4)
rows = [k for k in range(256) if occ[k] == 4 and spin[k] == 0]
reference = {0.0: 0.63554847, 0.05: 0.45058234,
             0.20: -0.18348455, 0.50: -1.69173670}

print(f"{'f':>6s} {'dim':>5s} {'E_0 (Jordan-Wigner)':>21s} {'chapter 4':>14s}")
for f, expected in reference.items():
    H = pairing_ph_full_space(4, g=1.0, f=f)
    E = np.sort(np.linalg.eigvalsh(H[np.ix_(rows, rows)]).real)
    print(f"{f:6.2f} {len(rows):5d} {E[0]:21.8f} {expected:14.8f}")

## Where this leads

Equation $\hat H = \sum_\alpha h_\alpha P_\alpha$ is the input to essentially
every quantum algorithm for many-body systems.

- In the **variational quantum eigensolver** the energy is obtained by
  measuring each Pauli string separately and adding the results with their
  weights, so the number of strings is the number of measurements.
- In a **Trotterised time evolution** each string contributes one factor
  $\exp(-i h_\alpha P_\alpha \Delta t)$ to the circuit, so the number of
  strings is the circuit depth and the weight of each is its entangling cost.

Both counts are what the tables above are really about, and both are the
reason it pays to exploit a symmetry — the seniority here — *before* choosing
an encoding rather than after.